In [1]:
# -*- coding: utf-8 -*-
"""
ttv_dataset_bounded_er.py
=========================
Extends ttv_dataset_with_er.py with physically-grounded Einstein radius
bounds [0.5, 2.0] arcsec, motivated by:

  • SLACS survey (Bolton et al. 2006, Auger et al. 2010, Shu et al. 2017)
        Observed range : [0.5, 1.8] arcsec  (85 grade-A lenses)
        Mean           : ~1.05 arcsec

  • Euclid population forecast (Collett 2015)
        Peak           : ~0.5 arcsec
        Detectable pop : [0.3, 2.0] arcsec; completeness floor ~0.5 arcsec

  • SIE multiple-imaging condition used in all substructure ML papers
        Hard minimum   : 0.5 arcsec  (Hezaveh et al. 2016, Vegetti et al. 2018)

  • Galaxy-scale lens upper limit
        Hard maximum   : 2.0 arcsec  (above this → cluster-scale regime)

═══════════════════════════════════════════════════════════════════════════════
WHAT CHANGED vs ttv_dataset_with_er.py
═══════════════════════════════════════════════════════════════════════════════

NEW imports
───────────
  from scipy.stats import truncnorm         ← truncated normal sampler

NEW module-level constants
──────────────────────────
  THETA_E_MIN  = 0.5   arcsec  (literature lower bound)
  THETA_E_MAX  = 2.0   arcsec  (literature upper bound)
  D_LS_OVER_DS = 0.636          (FlatLambdaCDM H0=70, Om0=0.3,
                                  z_lens=0.5, z_source_convention=2.5)
  C_KMS        = 2.998e5  km/s

NEW helper functions
────────────────────
  sample_theta_E(mean, std)
      Draws θ_E from TruncNormal(mean, std) bounded to [THETA_E_MIN, THETA_E_MAX].
      Used by both simulators so bounds are enforced in one place.

  sigma_v_from_theta_E(theta_E_arcsec)
      Analytically inverts the SIS lensing equation:
          θ_E [rad] = 4π (σ_v/c)² × (D_ls/D_s)
      → σ_v = c × √( θ_E_rad / (4π × D_ls/D_s) )
      Round-trip error verified < 1 micro-arcsec at all boundary values.

CHANGED: simulate_no_sub()
──────────────────────────
  BEFORE  sigma_v = np.random.normal(260, 20)
  AFTER   sigma_v = sigma_v_from_theta_E(sample_theta_E(mean=1.24, std=0.19))

  Why:  The original N(260,20) maps to θ_E ~ N(1.24, 0.19) arcsec; 99.97% of
        draws already fall in [0.5, 2.0].  The truncation is therefore nearly
        invisible to the distribution shape but eliminates the 0.03% tail that
        would produce unphysical sub-resolution or cluster-scale θ_E values.

CHANGED: simulator_model_no_sub()  AND  simulator_model_cdm()
──────────────────────────────────────────────────────────────
  BEFORE  Ens = np.random.normal(1.5, 0.5)
  AFTER   Ens = sample_theta_E(mean=1.5, std=0.5)

  For No_Sub:  Ens is overridden inside simulate_no_sub by the JOSEPH block
               (sigma_v conversion), so this change is cosmetic — but it
               prevents Ens from going unphysically negative or >2.5.
  For CDM:     Ens IS the true θ_E, so enforcing [0.5, 2.0] here is the
               primary physics fix for CDM.

Distribution summary after changes
───────────────────────────────────
  Class    Sampler                          Mean    Std    Range
  ───────  ──────────────────────────────── ─────── ────── ─────────────
  CDM      TruncNormal(1.5, 0.5, 0.5, 2.0) 1.385″  0.361″ [0.50, 2.00]″
  No_Sub   TruncNormal(1.24, 0.19, 0.5, 2.0)1.241″ 0.190″ [0.50, 2.00]″

  CDM mean shifts 1.50 → 1.39 because the upper tail (>2.0) is removed;
  No_Sub is essentially unchanged (0.03% truncation).

All other logic (DatasetGenerator, checkpointing, er_{split}.npy output)
is identical to ttv_dataset_with_er.py.
═══════════════════════════════════════════════════════════════════════════════
PATCH NOTES (v2, for the theta_E / SSIM checks)
  1. Saves src_{split}.npy: the unlensed, PSF-convolved, max-normalised source on the same 64x64 grid
     (render_unlensed_source). Needed for the source-reconstruction SSIM.
  2. sigma_v_from_theta_E now uses lenstronomy's LensCosmo. The old constant D_ls/D_s = 0.636 disagrees with
     lenstronomy (0.676 for this cosmology and z_lens=0.5, z_source=2.5), so sampled theta_E came out ~6%
     too large (up to ~2.13 arcsec). Saved labels were always read back from lenstronomy and are correct.
  3. PHYS_ONLY = True (default) generates only a 1,500-image test split into ...-Eins-Phys; the existing
     ...-Eins train/val/test keep their er_*.npy labels and are not touched.
  Images generated before this patch get NaN source frames when resumed; eval_physics drops them.

"""

from google.colab import drive
drive.mount('/content/drive')

# ── Installations ──────────────────────────────────────────────────────────
import subprocess
# pyHalo 1.4.7 and later import lenstronomy profiles (core_collapsed_halo_bh, king) that are not in any
# released lenstronomy (latest on PyPI is 1.14.2). 1.4.6 is the newest pyHalo that imports cleanly.
# If you see the old ModuleNotFoundError after this cell, restart the runtime and run it again.
for pkg in ['lenstronomy', 'pyHalo==1.4.6', 'colossus', 'mcfit']:
    subprocess.run(['pip', 'install', pkg, '-q'], check=False)

# ── Standard imports ───────────────────────────────────────────────────────
import os
import copy
import json
import pickle
import random
from datetime import datetime

import h5py
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from scipy.stats import truncnorm          # ← NEW: truncated normal sampler
from tqdm import tqdm
from IPython.display import clear_output

# ── lenstronomy ────────────────────────────────────────────────────────────
import lenstronomy.Util.data_util as data_util
import lenstronomy.Util.util as util
import lenstronomy.Plots.plot_util as plot_util
from lenstronomy.SimulationAPI.sim_api import SimAPI
from lenstronomy.LightModel.Profiles.gaussian import GaussianEllipse
from lenstronomy.LensModel.Profiles.cnfw import CNFW
from lenstronomy.LensModel.Profiles.nfw import NFW
from lenstronomy.LensModel.lens_model import LensModel
from lenstronomy.LensModel.lens_model_extensions import LensModelExtensions
from lenstronomy.ImSim.image_model import ImageModel          # PATCH: unlensed source renderer
from lenstronomy.LightModel.light_model import LightModel     # PATCH
from lenstronomy.Cosmo.lens_cosmo import LensCosmo           # PATCH: exact D_ls/D_s from the same cosmology
gauss = GaussianEllipse()

# ── Cosmology ──────────────────────────────────────────────────────────────
from astropy.cosmology import FlatLambdaCDM, FLRW
import inspect as _inspect

# Compatibility shim. lenstronomy 1.14.x calls cosmo.angular_diameter_distance(z1, z2), a two-argument form
# that only exists in astropy >= 8. Older astropy (Colab ships 7.x) raises
# "takes 2 positional arguments but 3 were given". Patched on the FLRW base class because pyHalo builds
# its own cosmology objects internally (realizations.astropy_instance), so patching `cosmo` alone is not enough.
_orig_add = FLRW.angular_diameter_distance
if len(_inspect.signature(_orig_add).parameters) < 3:
    def _angular_diameter_distance(self, z, z2=None):
        if z2 is None:
            return _orig_add(self, z)
        return self.angular_diameter_distance_z1z2(z, z2)
    FLRW.angular_diameter_distance = _angular_diameter_distance

cosmo = FlatLambdaCDM(H0=70, Om0=0.3, Ob0=0.)

# ── Euclid instrument config ───────────────────────────────────────────────
from lenstronomy.SimulationAPI.ObservationConfig.Euclid import Euclid
Euclid_g = Euclid(band='VIS', psf_type='GAUSSIAN', coadd_years=6)
kwargs_g_band = Euclid_g.kwargs_single_band()

# ── pyHalo ─────────────────────────────────────────────────────────────────
from pyHalo.PresetModels.cdm import CDM
from pyHalo.PresetModels.uldm import ULDM
from pyHalo.Halos.lens_cosmo import LensCosmo as PyHaloLensCosmo   # alias: must not shadow lenstronomy's LensCosmo
from pyHalo.single_realization import SingleHalo


# ═══════════════════════════════════════════════════════════════════════════
# NEW: Physics constants and literature-grounded bounds
# ═══════════════════════════════════════════════════════════════════════════

THETA_E_MIN  = 0.5      # arcsec — SLACS/Euclid/ML detection floor
THETA_E_MAX  = 2.0      # arcsec — galaxy-scale lens upper limit

# Angular diameter distance ratio D_ls / D_s
# Computed from FlatLambdaCDM(H0=70, Om0=0.3), z_lens=0.5,
# z_source_convention=2.5 (matches kwargs_model_physical in the code)
#   D_A(z=0.5)   ≈ 1259 Mpc
#   D_A(z=2.5)   ≈ 1480 Mpc
#   D_A(0.5→2.5) ≈  941 Mpc
#   D_ls / D_s   = 941 / 1480 ≈ 0.636
D_LS_OVER_DS = 0.636   # NOT USED any more (see PATCH below): lenstronomy's own conversion gives 0.676 for
                       # FlatLambdaCDM(H0=70, Om0=0.3), z_lens=0.5, z_source=2.5, so 0.636 made every sampled
                       # theta_E ~6% too large (max ~2.13", not 2.0"). The saved labels were always read back from
                       # lenstronomy, so they are correct; only the sampling distribution was shifted.
_LENS_COSMO = LensCosmo(z_lens=0.5, z_source=2.5, cosmo=cosmo)   # PATCH

C_KMS = 2.998e5  # speed of light in km/s


def sample_theta_E(mean, std):
    """
    Sample a single Einstein radius from TruncNormal(mean, std)
    bounded strictly to [THETA_E_MIN, THETA_E_MAX] = [0.5, 2.0] arcsec.

    Literature basis
    ────────────────
    • SLACS observed range [0.5, 1.8]″  (Bolton 2006, Auger 2010, Shu 2017)
    • Euclid detectable population [0.3, 2.0]″, completeness floor ~0.5″
      (Collett 2015)
    • SIE multiple-imaging detectability minimum ~0.5″
      (Hezaveh et al. 2016, Vegetti et al. 2018)
    • Galaxy-scale vs cluster-scale boundary ~2.0″

    Parameters
    ──────────
    mean : float  — distribution centre (arcsec)
    std  : float  — distribution width (arcsec)

    Returns
    ───────
    float — θ_E in arcsec, guaranteed in [THETA_E_MIN, THETA_E_MAX]
    """
    a = (THETA_E_MIN - mean) / std   # lower standardised bound
    b = (THETA_E_MAX - mean) / std   # upper standardised bound
    return float(truncnorm.rvs(a, b, loc=mean, scale=std))


def sigma_v_from_theta_E(theta_E_arcsec):
    """
    Analytically invert the SIS lensing equation to recover the velocity
    dispersion σ_v (km/s) corresponding to a target Einstein radius.

    SIS formula (Schneider, Ehlers & Falco 1992):
        θ_E [rad] = 4π (σ_v / c)² × (D_ls / D_s)

    Inverted:
        σ_v = c × √( θ_E_rad / (4π × D_ls/D_s) )

    Uses module-level D_LS_OVER_DS = 0.636, matched to the cosmology and
    redshifts configured in kwargs_model_physical (z_lens=0.5, z_src=2.5).

    Round-trip accuracy (verified):
        θ_E → σ_v → θ_E back:  error < 1 micro-arcsec at all boundary values

    Parameters
    ──────────
    theta_E_arcsec : float  — target Einstein radius (arcsec)

    Returns
    ───────
    float — velocity dispersion σ_v in km/s

    Boundary reference
    ──────────────────
        θ_E = 0.5″  →  σ_v ≈ 165 km/s
        θ_E = 1.0″  →  σ_v ≈ 234 km/s
        θ_E = 1.24″ →  σ_v ≈ 260 km/s  (original code mean)
        θ_E = 1.5″  →  σ_v ≈ 286 km/s
        θ_E = 2.0″  →  σ_v ≈ 330 km/s
    """
    # PATCH: use lenstronomy's own SIS conversion so sigma_v -> theta_E round-trips exactly
    return float(_LENS_COSMO.sis_theta_E2sigma_v(theta_E_arcsec))


# ═══════════════════════════════════════════════════════════════════════════
# Real galaxy data loading  (unchanged)
# ═══════════════════════════════════════════════════════════════════════════

subprocess.run([
    'wget',
    'https://astro.utoronto.ca/~hleung/shared/Galaxy10/Galaxy10_DECals.h5',
    '-O', 'Galaxy10_DECals.h5', '--no-check-certificate'
], check=False)

with h5py.File('/content/Galaxy10_DECals.h5', 'r') as F:
    images = np.array(F['images'])
    typ    = np.array(F['ans'])
    z      = np.array(F['redshift'])

unbarred_spiral = np.where(typ == 6)
images_ref = images[unbarred_spiral]
z_ref      = z[unbarred_spiral]
indx_img_zp1 = np.where(z_ref < 0.02)
img_zp1      = images_ref[indx_img_zp1]

arr = [2,3,5,6,7,8,9,10,11,12,13,14,15,16,17,19,20,21,22,23,24,25,26,27,28,
       29,30,31,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,54,55]


# ── Helpers ────────────────────────────────────────────────────────────────

def gaus2d(x=0, y=0, mx=0, my=0, sx=1, sy=1):
    return (1. / (2. * np.pi * sx * sy)
            * np.exp(-((x-mx)**2./(2.*sx**2.) + (y-my)**2./(2.*sy**2.))))


# ═══════════════════════════════════════════════════════════════════════════
# PATCH: ground-truth source for the SSIM check
# ═══════════════════════════════════════════════════════════════════════════

def render_unlensed_source(sim_g, kwargs_source_g, kwargs_numerics):
    """
    The same source galaxy, same pixel grid, same PSF, no lens, no noise, no lens light.
    Max-normalised, so it is directly comparable with the max-normalised source image that the
    SIS layer returns. Shape (64, 64), float32.
    """
    nolens = ImageModel(data_class=sim_g.data_class, psf_class=sim_g.psf_class,
                        lens_model_class=LensModel([]),
                        source_model_class=LightModel(['INTERPOL']),
                        kwargs_numerics=kwargs_numerics)
    src = nolens.image(kwargs_lens=[], kwargs_source=kwargs_source_g,
                       source_add=True, lens_light_add=False, point_source_add=False)
    src = np.clip(np.asarray(src, dtype=np.float32), 0, None)
    return src / max(float(src.max()), 1e-12)


# ═══════════════════════════════════════════════════════════════════════════
# NO-SUB SIMULATOR
# ═══════════════════════════════════════════════════════════════════════════

def simulate_no_sub(image_galaxy, axion_mass, Ens,
                    source_pos_xx, source_pos_yy, source_ang,
                    zlens=0.5, zsource=1.0):
    """
    Simulate a no-substructure (ULDM) lensing image.

    θ_E control:
        The JOSEPH block overrides Ens and drives θ_E via σ_v.
        CHANGE: σ_v is now derived by inverting a θ_E sample drawn from
        TruncNormal(1.24, 0.19, [0.5, 2.0]):
            1.  θ_E_target = sample_theta_E(mean=1.24, std=0.19)
            2.  σ_v        = sigma_v_from_theta_E(θ_E_target)
            3.  θ_E_actual = kwargs_lens[0]['theta_E']  ← lenstronomy output
        θ_E_actual ≈ θ_E_target (error < 1 μas), guaranteed in [0.5, 2.0]″.

    Returns (image_g, realizationsULDM, theta_E, src_gt)   # PATCH: src_gt added
    """
    axion_mass = -21.5

    profile_arg = {
        'log10_m_uldm': axion_mass,
        'uldm_plaw': 1/3,
        'scale_nfw': False,
        'evaluate_mc_at_zlens': True,
        'c_scatter': False
    }

    main_halo_type   = 'SIE'
    kwargs_lens_main = {'theta_E': Ens, 'e1': 0, 'e2': 0,
                        'center_x': 0, 'center_y': 0.}
    kwargs_shear     = {'gamma1': 0.05, 'gamma2': 0}
    lens_model_macro = [main_halo_type, 'SHEAR']
    kwargs_lens      = [kwargs_lens_main, kwargs_shear]

    log10_m_uldms, uldm_plaw, scale_nfw = axion_mass, 1/3, False
    realizationsULDM = ULDM(zlens, zsource,
                            log10_m_uldm=log10_m_uldms,
                            uldm_plaw=uldm_plaw,
                            flucs=scale_nfw)
    astropy_instance = realizationsULDM.astropy_instance

    # ── JOSEPH block ───────────────────────────────────────────────────────
    kwargs_model_physical = {
        'lens_model_list': ['SIE'],
        'lens_redshift_list': [0.5],
        'lens_light_model_list': ['SERSIC_ELLIPSE', 'SERSIC_ELLIPSE'],
        'source_light_model_list': ['INTERPOL'],
        'source_redshift_list': [1.0],
        'cosmo': cosmo,
        'z_source_convention': 2.5,
        'z_source': 2.5,
    }

    # ── CHANGE: sample θ_E in [0.5, 2.0], back-compute σ_v ───────────────
    # Original: sigma_v = np.random.normal(260, 20)
    # Replaced: sample θ_E from TruncNormal(1.24, 0.19), then invert SIS
    #
    # Why mean=1.24, std=0.19?
    #   The original N(260, 20) maps exactly to N(1.24, 0.19) in θ_E space
    #   via θ_E = 4π(σ_v/c)² × D_ls/D_s.  We preserve the same distribution
    #   shape and only add the [0.5, 2.0] hard bounds (which clip 0.03% of
    #   the original draws).
    theta_E_target = sample_theta_E(mean=1.24, std=0.19)
    sigma_v        = sigma_v_from_theta_E(theta_E_target)

    kwargs_mass = [{'sigma_v': sigma_v,
                    'center_x': 0, 'center_y': 0,
                    'e1': 0.0, 'e2': 0}]

    kwargs_model_postit = kwargs_model_physical

    numpix          = 64
    kwargs_numerics = {'point_source_supersampling_factor': 1}

    sim_g   = SimAPI(numpix,
                     kwargs_single_band=kwargs_g_band,
                     kwargs_model=kwargs_model_postit)
    imSim_g = sim_g.image_model_class(kwargs_numerics)

    source_scale = 0.015
    X, Y = source_pos_xx, source_pos_yy

    image_data_g  = image_galaxy.astype(float)
    median_g      = np.median(image_galaxy[:50, :50].astype(float))
    image_data_g -= median_g

    kwargs_lens_light_mag_g = [
        {'magnitude': 40, 'R_sersic': 0.4, 'n_sersic': 2.3,
         'e1': 0, 'e2': 0.05, 'center_x': 0, 'center_y': 0},
        {'magnitude': 40, 'R_sersic': 1.5, 'n_sersic': 1.2,
         'e1': 0, 'e2': 0.3,  'center_x': 0, 'center_y': 0}
    ]
    kwargs_source_mag_g = [
        {'magnitude': np.random.normal(16, .1),
         'image': image_data_g, 'scale': source_scale,
         'phi_G': source_ang, 'center_x': X, 'center_y': Y}
    ]

    kwargs_lens_light_g, kwargs_source_g, _ = sim_g.magnitude2amplitude(
        kwargs_lens_light_mag_g, kwargs_source_mag_g
    )

    # physical2lensing_conversion uses sigma_v → theta_E internally
    # kwargs_lens[0]['theta_E'] will equal theta_E_target to < 1 μas
    kwargs_lens = sim_g.physical2lensing_conversion(kwargs_mass=kwargs_mass)

    # Ground-truth θ_E: read from lenstronomy output (not from theta_E_target)
    # so the saved value is exactly what the raytracing used
    theta_E = float(kwargs_lens[0]['theta_E'])

    image_g  = imSim_g.image(kwargs_lens, kwargs_source_g, kwargs_lens_light_g)
    image_g += sim_g.noise_for_model(model=image_g)
    src_gt   = render_unlensed_source(sim_g, kwargs_source_g, kwargs_numerics)   # PATCH

    return image_g, realizationsULDM, theta_E, src_gt


def simulator_model_no_sub(parameter, img_zp1, return_points=False):
    """
    CHANGE: Ens now sampled from TruncNormal(1.5, 0.5, [0.5, 2.0]).

    Note: Ens is overridden inside simulate_no_sub by the σ_v → θ_E path,
    so the physics is not affected.  The bounded sampling prevents Ens from
    being negative or anomalously large before the override.

    Returns (sim, parameter, mask, orig, theta_E, src_gt)   # PATCH
    """
    index  = np.random.randint(0, len(img_zp1))

    # ── CHANGE: bounded Ens (cosmetically bounded; overridden in simulate_no_sub)
    # Original: Ens = np.random.normal(1.5, .5)
    Ens           = sample_theta_E(mean=1.5, std=0.5)
    source_pos_xx = np.random.uniform(-0.4, 0.4)
    source_pos_yy = np.random.uniform(-0.4, 0.4)
    source_ang    = np.random.uniform(-np.pi, np.pi)

    x  = np.linspace(-128, 128, 256)
    y  = np.linspace(-128, 128, 256)
    result = gaussian_filter(img_zp1[index][:, :, 0], sigma=3)
    mx, my, sx, sy = 0, 0, 25, 25
    xx, yy = np.meshgrid(x, y)
    zz = gaus2d(x=xx, y=yy, mx=mx, my=my, sx=sx, sy=sy)

    sim, realizationsULDM, theta_E, src_gt = simulate_no_sub(
        result, parameter, Ens,
        source_pos_xx, source_pos_yy, source_ang
    )

    return np.absolute(sim), parameter, zz * result, result, theta_E, src_gt


def simulate_image_no_sub(img_zp1):
    """
    Returns (image, theta_E, src_gt). theta_E is read from lenstronomy; src_gt is the unlensed source.
    """
    amass = -21.5
    sim, mass, mask, orig, theta_E, src_gt = simulator_model_no_sub(amass, img_zp1)
    return np.array(sim).clip(min=0), theta_E, src_gt


# ═══════════════════════════════════════════════════════════════════════════
# CDM SIMULATOR
# ═══════════════════════════════════════════════════════════════════════════

def simulate_cdm(image_galaxy, Ens,
                 source_pos_xx, source_pos_yy, source_ang,
                 zlens=0.5, zsource=1.0):
    """
    Simulate a CDM lensing image.

    θ_E control:
        Ens is the direct θ_E — no conversion happens.
        Caller (simulator_model_cdm) now draws Ens from
        TruncNormal(1.5, 0.5, [0.5, 2.0]), so Ens is guaranteed in bounds.

    Returns (image_g, realizationsCDM, theta_E, src_gt)   # PATCH
    """
    main_halo_type   = 'SIE'
    kwargs_lens_main = {'theta_E': Ens, 'e1': 0, 'e2': 0,
                        'center_x': 0, 'center_y': 0.}
    kwargs_shear     = {'gamma1': 0.05, 'gamma2': 0}
    lens_model_macro = [main_halo_type, 'SHEAR']
    kwargs_lens      = [kwargs_lens_main, kwargs_shear]

    realizationsCDM = CDM(zlens, zsource, cone_opening_angle_arcsec=10)

    lens_model_list, lens_redshift_array, kwargs_halos, _ = \
        realizationsCDM.lensing_quantities()
    astropy_instance = realizationsCDM.astropy_instance

    lens_model_list_full     = lens_model_macro + lens_model_list
    lens_redshift_list_full  = [zlens, zlens] + list(lens_redshift_array)
    kwargs_lens_full         = kwargs_lens + kwargs_halos

    kwargs_model_physical = {
        'lens_model_list': lens_model_list_full,
        'lens_redshift_list': lens_redshift_list_full,
        'lens_light_model_list': ['SERSIC_ELLIPSE', 'SERSIC_ELLIPSE'],
        'source_light_model_list': ['INTERPOL'],
        'source_redshift_list': [1.0],
        'cosmo': astropy_instance,
        'z_source_convention': 2.5,
        'z_source': 2.5,
    }

    numpix          = 64
    kwargs_numerics = {'point_source_supersampling_factor': 1}

    sim_g   = SimAPI(numpix,
                     kwargs_single_band=kwargs_g_band,
                     kwargs_model=kwargs_model_physical)
    imSim_g = sim_g.image_model_class(kwargs_numerics)

    source_scale = 0.015
    X, Y = source_pos_xx, source_pos_yy

    image_data_g  = image_galaxy.astype(float)
    median_g      = np.median(image_galaxy[:50, :50].astype(float))
    image_data_g -= median_g

    kwargs_lens_light_mag_g = [
        {'magnitude': 40, 'R_sersic': 0.4, 'n_sersic': 2.3,
         'e1': 0, 'e2': 0.05, 'center_x': 0, 'center_y': 0},
        {'magnitude': 40, 'R_sersic': 1.5, 'n_sersic': 1.2,
         'e1': 0, 'e2': 0.3,  'center_x': 0, 'center_y': 0}
    ]
    kwargs_source_mag_g = [
        {'magnitude': np.random.normal(16, 0.1),
         'image': image_data_g, 'scale': source_scale,
         'phi_G': source_ang, 'center_x': X, 'center_y': Y}
    ]

    kwargs_lens_light_g, kwargs_source_g, _ = sim_g.magnitude2amplitude(
        kwargs_lens_light_mag_g, kwargs_source_mag_g
    )

    # No physical2lensing_conversion — Ens is already in angular units
    image_g  = imSim_g.image(kwargs_lens_full, kwargs_source_g,
                              kwargs_lens_light_g)
    image_g += sim_g.noise_for_model(model=image_g)
    src_gt   = render_unlensed_source(sim_g, kwargs_source_g, kwargs_numerics)   # PATCH

    return image_g, realizationsCDM, float(Ens), src_gt


def simulator_model_cdm(img_zp1, return_points=False):
    """
    CHANGE: Ens drawn from TruncNormal(1.5, 0.5, [0.5, 2.0]) — primary fix
    for CDM class since Ens is the literal θ_E used in raytracing.

    Before: Ens = np.random.normal(1.5, 0.5)  — unbounded, ~2.3% > 2.0″
    After : Ens = sample_theta_E(mean=1.5, std=0.5)  — hard-capped at 2.0″

    Returns (sim, mask, orig, theta_E, src_gt)   # PATCH
    """
    index  = np.random.randint(0, len(img_zp1))

    # ── CHANGE: bounded sampling ──────────────────────────────────────────
    # Original: Ens = np.random.normal(1.5, .5)
    # Replaces: unbounded Gaussian with TruncNormal bounded to [0.5, 2.0]″
    # Literature: SLACS range [0.5, 1.8]″ (Auger 2010); Euclid upper tail
    #             up to ~2.0″ (Collett 2015); cluster-scale begins >2.0″
    Ens           = sample_theta_E(mean=1.5, std=0.5)
    source_pos_xx = np.random.uniform(-0.4, 0.4)
    source_pos_yy = np.random.uniform(-0.4, 0.4)
    source_ang    = np.random.uniform(-np.pi, np.pi)

    x  = np.linspace(-128, 128, 256)
    y  = np.linspace(-128, 128, 256)
    result = gaussian_filter(img_zp1[index][:, :, 0], sigma=3)
    mx, my, sx, sy = 0, 0, 25, 25
    xx, yy = np.meshgrid(x, y)
    zz = gaus2d(x=xx, y=yy, mx=mx, my=my, sx=sx, sy=sy)

    sim, realizationsCDM, theta_E, src_gt = simulate_cdm(
        result, Ens,
        source_pos_xx, source_pos_yy, source_ang
    )

    return np.absolute(sim), zz * result, result, theta_E, src_gt


def simulate_image_cdm(img_zp1):
    """
    Returns (image, theta_E, src_gt).
    """
    sim, mask, orig, theta_E, src_gt = simulator_model_cdm(img_zp1)
    return np.array(sim).clip(min=0), theta_E, src_gt


# ═══════════════════════════════════════════════════════════════════════════
# DATASET GENERATOR  (unchanged from ttv_dataset_with_er.py)
# ═══════════════════════════════════════════════════════════════════════════

class DatasetGenerator:
    """
    Generates strong lensing simulations with checkpointing.
    Saves data_{split}.npy, labels_{split}.npy, er_{split}.npy.

    er_{split}.npy contains ground-truth Einstein radii in arcsec,
    all guaranteed in [THETA_E_MIN, THETA_E_MAX] = [0.5, 2.0].
    """

    def __init__(self, data_path='../data/Real-Galaxy-Tiny Dataset',
                 checkpoint_freq=5):
        self.data_path       = data_path
        self.checkpoint_freq = checkpoint_freq
        self.checkpoint_file = os.path.join(data_path, 'checkpoint.json')
        os.makedirs(data_path, exist_ok=True)
        self.checkpoint      = self.load_checkpoint()

    def load_checkpoint(self):
        if os.path.exists(self.checkpoint_file):
            try:
                with open(self.checkpoint_file, 'r') as f:
                    checkpoint = json.load(f)
                print(f"✅ Loaded checkpoint: "
                      f"{checkpoint['completed_images']} images completed")
                if 'val' not in checkpoint['splits']:
                    checkpoint['splits'].append('val')
                    checkpoint['split_progress']['val'] = 0
                if 'er_progress' not in checkpoint:
                    checkpoint['er_progress'] = {
                        s: 0 for s in checkpoint['splits']
                    }
                return checkpoint
            except Exception as e:
                print(f"⚠️ Error loading checkpoint: {e}. Starting fresh.")

        return {
            'completed_images': 0,
            'current_split': 'train',
            'target_images': {'train': 0, 'test': 0, 'val': 0},
            'splits': ['train', 'test', 'val'],
            'timestamp': datetime.now().isoformat(),
            'split_progress': {'train': 0, 'test': 0, 'val': 0},
            'er_progress':    {'train': 0, 'test': 0, 'val': 0},
        }

    def save_checkpoint(self):
        self.checkpoint['timestamp'] = datetime.now().isoformat()
        try:
            with open(self.checkpoint_file, 'w') as f:
                json.dump(self.checkpoint, f, indent=2)
        except Exception as e:
            print(f"⚠️ Error saving checkpoint: {e}")

    def load_existing_data(self, split):
        data_file   = os.path.join(self.data_path, f'data_{split}.npy')
        labels_file = os.path.join(self.data_path, f'labels_{split}.npy')
        er_file     = os.path.join(self.data_path, f'er_{split}.npy')
        src_file    = os.path.join(self.data_path, f'src_{split}.npy')    # PATCH

        if os.path.exists(data_file) and os.path.exists(labels_file):
            try:
                data   = np.load(data_file).tolist()
                labels = np.load(labels_file).tolist()
                print(f"📂 Loaded existing {split} data: {len(data)} images")
                if os.path.exists(er_file):
                    er = np.load(er_file).tolist()
                    print(f"📂 Loaded existing {split} Einstein radii: {len(er)}")
                else:
                    er = [float('nan')] * len(data)
                    print(f"⚠️  No er_{split}.npy. Filling {len(er)} with NaN.")
                # PATCH: ground-truth sources; NaN frames mark images generated before this patch
                if os.path.exists(src_file):
                    src = list(np.load(src_file))
                    print(f"📂 Loaded existing {split} source images: {len(src)}")
                else:
                    src = [np.full((64, 64), np.nan, dtype=np.float32) for _ in range(len(data))]
                    print(f"⚠️  No src_{split}.npy. Filling {len(src)} frames with NaN.")
                return data, labels, er, src
            except Exception as e:
                print(f"⚠️ Error loading existing data: {e}")
                return [], [], [], []
        return [], [], [], []

    def save_data_arrays(self, data, labels, einstein_radii, sources, split):
        try:
            np.save(os.path.join(self.data_path, f'data_{split}.npy'),
                    np.array(data))
            np.save(os.path.join(self.data_path, f'labels_{split}.npy'),
                    np.array(labels))
            np.save(os.path.join(self.data_path, f'er_{split}.npy'),
                    np.array(einstein_radii, dtype=np.float32))
            np.save(os.path.join(self.data_path, f'src_{split}.npy'),          # PATCH
                    np.array(sources, dtype=np.float32))
            print(f"💾 Saved {len(data)} images + "
                  f"{len(einstein_radii)} Einstein radii → {split}")
        except Exception as e:
            print(f"❌ Error saving data: {e}")

    def generate_dataset(self, img_zp1, target_images):
        """
        Generate dataset.  Every θ_E in er_{split}.npy is in [0.5, 2.0]″.

        File layout:
            data_{split}.npy    float32 [N, 64, 64]   lensed images
            labels_{split}.npy  int     [N, 2]         one-hot labels
            er_{split}.npy      float32 [N]            θ_E in arcsec

        Labels:
            [1, 0] → ULDM / no-substructure
            [0, 1] → CDM
        """
        self.checkpoint['target_images'] = target_images

        for split in self.checkpoint['splits']:
            target = self.checkpoint['target_images'].get(split, 0)
            if target == 0 or (self.checkpoint['split_progress'][split] * 2
                                >= target):
                print(f"✅ Split '{split}' already complete or no target.")
                continue

            print(f"\n🚀 Processing split: {split}")
            data, labels, einstein_radii, sources = self.load_existing_data(split)   # PATCH

            current_iterations   = len(data) // 2
            remaining_iterations = (target // 2) - current_iterations

            if remaining_iterations <= 0:
                print(f"✅ Split '{split}' already has enough images")
                self.checkpoint['split_progress'][split] = target // 2
                continue

            print(f"📊 {current_iterations}/{target // 2} iterations done")
            print(f"🎯 Generating {remaining_iterations} more…")

            pbar = tqdm(range(remaining_iterations),
                        desc=f"Generating {split}")
            try:
                for i in pbar:
                    no_sub_img, no_sub_er, no_sub_src = simulate_image_no_sub(img_zp1)   # PATCH
                    no_sub_img = no_sub_img / np.max(no_sub_img)

                    cdm_img, cdm_er, cdm_src = simulate_image_cdm(img_zp1)   # PATCH
                    cdm_img = cdm_img / np.max(cdm_img)

                    data.append(no_sub_img.reshape(64, 64))
                    labels.append([1, 0])
                    data.append(cdm_img.reshape(64, 64))
                    labels.append([0, 1])

                    einstein_radii.append(float(no_sub_er))
                    einstein_radii.append(float(cdm_er))
                    sources.append(no_sub_src.reshape(64, 64))      # PATCH: same order as data / labels / er
                    sources.append(cdm_src.reshape(64, 64))

                    current_iteration = current_iterations + i + 1
                    self.checkpoint['split_progress'][split] = current_iteration
                    self.checkpoint['er_progress'][split]    = len(einstein_radii)
                    self.checkpoint['completed_images']      = sum(
                        self.checkpoint['split_progress'].values()
                    ) * 2

                    if current_iteration % self.checkpoint_freq == 0:
                        self.save_data_arrays(data, labels, einstein_radii, sources, split)
                        self.save_checkpoint()
                        pbar.set_postfix({
                            'imgs': len(data),
                            'θ_E': f'{len(einstein_radii)} saved',
                        })

                self.save_data_arrays(data, labels, einstein_radii, sources, split)
                self.save_checkpoint()
                print(f"✅ '{split}': {len(data)} images, "
                      f"{len(einstein_radii)} Einstein radii")

            except KeyboardInterrupt:
                print(f"\n⏸️ Interrupted. Saving…")
                self.save_data_arrays(data, labels, einstein_radii, sources, split)
                self.save_checkpoint()
                self.print_status()
                break

            except Exception as e:
                print(f"\n❌ Error: {e}")
                self.save_data_arrays(data, labels, einstein_radii, sources, split)
                self.save_checkpoint()
                raise

            clear_output(wait=True)

        print("\n🎉 Dataset generation completed!")
        self.print_status()
        if self.is_complete():
            self.cleanup_checkpoint()

    def print_status(self):
        print("\n📊 DATASET STATUS")
        print("=" * 55)
        total_target = sum(self.checkpoint['target_images'].values())
        for split in self.checkpoint['splits']:
            prog   = self.checkpoint['split_progress'].get(split, 0)
            target = self.checkpoint['target_images'].get(split, 0)
            er_cnt = self.checkpoint['er_progress'].get(split, 0)
            pct    = (prog * 2 / target * 100) if target > 0 else 0
            print(f"{split:>5}: {prog:>4}/{target//2} iters  "
                  f"({prog*2:>4} imgs, {er_cnt:>4} θ_E ∈ [0.5,2.0]″)  "
                  f"[{pct:>5.1f}%]")
        total = self.checkpoint['completed_images']
        pct_t = (total / total_target * 100) if total_target > 0 else 0
        print("-" * 55)
        print(f"TOTAL: {total}/{total_target} [{pct_t:.1f}%]")
        print(f"Last: {self.checkpoint['timestamp']}")

    def is_complete(self):
        for split in self.checkpoint['splits']:
            target = self.checkpoint['target_images'].get(split, 0)
            if (target > 0
                    and self.checkpoint['split_progress'].get(split, 0) * 2
                    < target):
                return False
        return True

    def cleanup_checkpoint(self):
        try:
            if os.path.exists(self.checkpoint_file):
                os.remove(self.checkpoint_file)
                print("🧹 Cleaned up checkpoint")
        except Exception as e:
            print(f"⚠️ Could not remove checkpoint: {e}")

    def resume_generation(self, img_zp1):
        if self.is_complete():
            print("✅ Already complete!")
            self.print_status()
            return
        print("🔄 Resuming…")
        self.print_status()
        self.generate_dataset(img_zp1, self.checkpoint['target_images'])


# ═══════════════════════════════════════════════════════════════════════════
# ENTRY POINT
# ═══════════════════════════════════════════════════════════════════════════

def generate_dataset_with_checkpoints(
        img_zp1,
        data_path='../data/Real-Galaxy-Tiny Dataset',
        target_images={'train': 7000, 'test': 1500, 'val': 1500},
        checkpoint_freq=50):

    generator = DatasetGenerator(data_path=data_path,
                                 checkpoint_freq=checkpoint_freq)

    if (generator.checkpoint['completed_images'] > 0
            and os.path.exists(generator.checkpoint_file)):
        print("🔄 Found existing progress!")
        generator.print_status()
        response = input("\nResume from checkpoint? (y/n): ").lower()
        if response == 'y':
            generator.resume_generation(img_zp1)
        else:
            print("Starting fresh…")
            generator = DatasetGenerator(data_path, checkpoint_freq)
            generator.generate_dataset(img_zp1, target_images)
    else:
        print("🚀 Starting fresh dataset generation…")
        generator.generate_dataset(img_zp1, target_images)


# ── CDM galaxy pool (unchanged) ────────────────────────────────────────────
images_ref    = images
z_ref         = z
indx_img_zp1  = np.where(z_ref < 1)
img_zp1       = images_ref[indx_img_zp1]

# PATCH ---------------------------------------------------------------------------------------------
# The existing ...-Eins folder already has train/val/test with er_{split}.npy but NO source images, and the
# random draws were not stored, so sources cannot be recovered. Generate a fresh TEST split with both labels
# in a new folder. The theta_E / SSIM checks only need a test set (about 1,500 simulations).
#   PHYS_ONLY = True : test split only (default)    PHYS_ONLY = False : full 7000/1500/1500 with sources
PHYS_ONLY = True
data_path = '/content/drive/My Drive/Real-Galaxy-Tiny-Dataset-Eins-Phys'
targets   = ({'train': 0, 'test': 1500, 'val': 0} if PHYS_ONLY
             else {'train': 7000, 'test': 1500, 'val': 1500})

generate_dataset_with_checkpoints(
    img_zp1,
    data_path=data_path,
    target_images=targets,
    checkpoint_freq=5
)


# ═══════════════════════════════════════════════════════════════════════════
# DOWNSTREAM USAGE & VERIFICATION
# ═══════════════════════════════════════════════════════════════════════════
"""
Quick sanity check after generation — run once to verify bounds:

    er_train = np.load('er_train.npy')
    assert er_train.min() >= 0.5,  f"Below floor: {er_train.min():.3f}"
    assert er_train.max() <= 2.0,  f"Above ceiling: {er_train.max():.3f}"
    print(f"θ_E stats — mean: {er_train.mean():.3f}, "
          f"std: {er_train.std():.3f}, "
          f"range: [{er_train.min():.3f}, {er_train.max():.3f}]")

    # Separate by class (labels are one-hot: [1,0]=ULDM, [0,1]=CDM)
    labels   = np.load('labels_train.npy')
    uldm_idx = np.where(labels[:, 0] == 1)[0]
    cdm_idx  = np.where(labels[:, 1] == 1)[0]

    print(f"ULDM θ_E: mean={er_train[uldm_idx].mean():.3f}, "
          f"std={er_train[uldm_idx].std():.3f}")
    print(f"CDM  θ_E: mean={er_train[cdm_idx].mean():.3f}, "
          f"std={er_train[cdm_idx].std():.3f}")

    # Wire into LensPINN training (from lenspinn_fixed.py):
    model, results = build_and_train(
        train_data, train_labels,
        test_data,  test_labels,
        train_er=er_train,
        test_er=np.load('er_test.npy'),
        mode='large',
        n_epochs=50
    )
    # k_min=0.5, k_max=2.0 in lenspinn_fixed.py — matches these bounds exactly
"""

✅ Split 'val' already complete or no target.

🎉 Dataset generation completed!

📊 DATASET STATUS
train:    0/0 iters  (   0 imgs,    0 θ_E ∈ [0.5,2.0]″)  [  0.0%]
 test:  750/750 iters  (1500 imgs, 1500 θ_E ∈ [0.5,2.0]″)  [100.0%]
  val:    0/0 iters  (   0 imgs,    0 θ_E ∈ [0.5,2.0]″)  [  0.0%]
-------------------------------------------------------
TOTAL: 1500/1500 [100.0%]
Last: 2026-10-11T07:18:39.950209
🧹 Cleaned up checkpoint


'\nQuick sanity check after generation — run once to verify bounds:\n\n    er_train = np.load(\'er_train.npy\')\n    assert er_train.min() >= 0.5,  f"Below floor: {er_train.min():.3f}"\n    assert er_train.max() <= 2.0,  f"Above ceiling: {er_train.max():.3f}"\n    print(f"θ_E stats — mean: {er_train.mean():.3f}, "\n          f"std: {er_train.std():.3f}, "\n          f"range: [{er_train.min():.3f}, {er_train.max():.3f}]")\n\n    # Separate by class (labels are one-hot: [1,0]=ULDM, [0,1]=CDM)\n    labels   = np.load(\'labels_train.npy\')\n    uldm_idx = np.where(labels[:, 0] == 1)[0]\n    cdm_idx  = np.where(labels[:, 1] == 1)[0]\n\n    print(f"ULDM θ_E: mean={er_train[uldm_idx].mean():.3f}, "\n          f"std={er_train[uldm_idx].std():.3f}")\n    print(f"CDM  θ_E: mean={er_train[cdm_idx].mean():.3f}, "\n          f"std={er_train[cdm_idx].std():.3f}")\n\n    # Wire into LensPINN training (from lenspinn_fixed.py):\n    model, results = build_and_train(\n        train_data, train_labels,\n